In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

# import zipfile as zp          # used for unzipping ppi files
# from pathlib import Path      # used to play with pathnames to save 
# from datetime import datetime # used to manipulate time :)


# import wradlib as wr          # used for having fun with radar data

# from PIL import Image         # used for creating gif loops
# import os                     # used for retrieving file names

In [ ]:
# FUNCTION
#pcolormesh but takes 1D X and Y coordinates for centres of the pixels

def pcolormeshC(x_centers, y_centers, z, ax=None,
                            shading='auto', **pcolor_kwargs):
    """
    Create a pcolormesh from a 2D array and 1D coordinate-center arrays.

    Parameters
    ----------
    x_centers : 1D array
        X coordinates of cell centers (length = number of columns in z)
    y_centers : 1D array
        Y coordinates of cell centers (length = number of rows in z)
    z : 2D array
        Data array with shape (len(y_centers), len(x_centers))
    ax : matplotlib.axes.Axes, optional
        Existing axis to draw on
    shading : str
        Passed to pcolormesh (default: 'auto')
    **pcolor_kwargs
        Extra kwargs passed to pcolormesh

    Returns
    -------
    pcm : QuadMesh
        The pcolormesh object
    """

    x_centers = np.asarray(x_centers)
    y_centers = np.asarray(y_centers)
    z = np.asarray(z)

    if z.shape != (len(y_centers), len(x_centers)):
        raise ValueError(
            f"z shape {z.shape} does not match "
            f"(len(y_centers), len(x_centers)) = "
            f"({len(y_centers)}, {len(x_centers)})"
        )

    # Convert centers -> edges
    def centers_to_edges(c):
        dc = np.diff(c)

        edges = np.empty(len(c) + 1)

        # Interior edges
        edges[1:-1] = c[:-1] + dc / 2

        # Extrapolate outer edges
        edges[0] = c[0] - dc[0] / 2
        edges[-1] = c[-1] + dc[-1] / 2

        return edges

    x_edges = centers_to_edges(x_centers)
    y_edges = centers_to_edges(y_centers)

    if ax is None:
        fig, ax = plt.subplots()

    pcm = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading=shading,
        **pcolor_kwargs
    )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    return pcm

In [ ]:
# THIS BLOCK LOADS IN A MONTH-LONG HOURLY BARRA DATA BLOCK FOR TEMPERATURE AT ALL* (MOST) PRESSURE LEVELS

# choose the year and the month you want to look at data for
year  = 2024
month = 2

# add leading zeros if neccessary to the year and month strings
YYYY = str(year).zfill(4)
MM   = str(month).zfill(2)

# pressure levels with temperature variables associated with them
PLevels = [1000, 950, 925, 850, 700, 600, 500, 400, 300] #, 200] # the kernel crashes whenever I include 200 hPa

# create an empty data structure carrying-bag
BarraDataStructs = {}

# loop through each pressure level and grab a lat-lon slice over Queensland 
for PLevel in PLevels:
    
    variable = 'ta' + str(PLevel)

    # where the BARRA data live
    BARRAfolder = f'/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/{variable}/latest/'
    BARRAfile = variable + f'_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_{YYYY}{MM}-{YYYY}{MM}.nc'
    BARRApath = BARRAfolder + BARRAfile

    ds = xr.open_dataset(BARRApath)

    # take that Queensland slice
    ds = ds.sel(
        lat=slice(-30, -10),
        lon=slice(140, 160)
    )

    # make sure the temperature variable is named the same in each data array
    ds = ds.rename({variable: "ta"})

    # store that slice away
    BarraDataStructs[PLevel] = ds

    
# combine all of the loaded temperature data arrays for each pressure level into one data array
datasets = [BarraDataStructs[Plevel] for Plevel in PLevels]

GrandTdata = xr.concat(
    datasets,
    dim=xr.DataArray(PLevels, dims="pressure", name="pressure")
)

# add attributes to the new pressure coordinate
GrandTdata['pressure'].attrs = {
    'long_name': 'pressure',
    'standard_name': 'pressure',
    'units': 'hPa',
    'axis': 'P'
}

In [ ]:
# GrandTdata.time[324] = 2024-02-14T12:00:00.000000000
# TIME I WANT FOR THE CASE STUDY

# GrandTdata.lat[80] = -21.12
# LATITUDE OF MACKAY

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
RangeViewer = pcolormeshC(GrandTdata.lon, GrandTdata.pressure, GrandTdata.ta[:,324,80,:]-273.15, ax=ax, cmap='nipy_spectral', vmin=-40, vmax=40)

ax.set_xlabel('Longitude [Degrees East]')
ax.set_ylabel('Pressure [hPa]')
plt.gca().invert_yaxis()

plt.colorbar(RangeViewer, ax=ax, label = 'Temperature [Celsius]')
plt.grid()

plt.show()